# Máquinas de Soporte Vectorial (Support Vector Machines - SVM)

## Introducción Teórica

Las Máquinas de Soporte Vectorial son algoritmos de aprendizaje supervisado que pueden utilizarse para clasificación, regresión y detección de outliers. En su forma más básica, SVM busca encontrar el hiperplano que maximiza el margen entre dos clases.

### Conceptos Clave

**Hiperplano de separación**: En un espacio de características, el hiperplano que divide las clases.

**Vectores de soporte**: Puntos de datos más cercanos al hiperplano que determinan la posición y orientación del margen.

**Margen**: Distancia entre el hiperplano y los vectores de soporte. SVM maximiza este margen.



In [ ]:
# ============================================================================
# IMPORTS Y CONFIGURACIÓN
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC, SVC
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.pipeline import make_pipeline

import warnings
warnings.filterwarnings("ignore")

# Configuración de estilo para gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline
plt.rcParams["axes.grid"] = False

In [ ]:
# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_dataset_up_down(size, seed=39, noise_ratio=0.0):
    """
    Genera un dataset donde la clase está determinada por el signo de Y.
    
    Parámetros:
    -----------
    size : int
        Número de muestras a generar
    seed : int
        Semilla para reproducibilidad
    noise_ratio : float
        Proporción de ruido (0.0 a 1.0)
    """
    np.random.seed(seed)
    x = np.random.poisson(5, size) * (np.random.randint(0, 2, size) * 2 - 1)
    y = (np.random.poisson(5, size) + 1) * (np.random.randint(0, 2, size) * 2 - 1)
    X = np.array(list(zip(x, y)))
    Y = (X[:, 1] > 0).astype(np.int8)
    
    # Añadir ruido si se especifica
    if noise_ratio > 0:
        noise = np.random.binomial(1, noise_ratio, Y.shape[0])
        Y = np.logical_xor(Y, noise).astype(np.int8)
    
    return X, Y


def plot_svm_decision_boundary(clf, X, Y, title=None, ax=None, show_support=True):
    """
    Visualiza la frontera de decisión de un clasificador SVM.
    """
    if ax is None:
        ax = plt.gca()
    
    # Graficar frontera de decisión
    DecisionBoundaryDisplay.from_estimator(
        clf,
        X,
        ax=ax,
        grid_resolution=200,
        plot_method="contour",
        colors="k",
        levels=[-1, 0, 1],
        alpha=0.5,
        linestyles=["--", "-", "--"],
    )
    
    # Graficar puntos
    scatter = ax.scatter(X[:, 0], X[:, 1], c=Y, s=50, cmap=plt.cm.Spectral, edgecolors='k', linewidth=0.0)
    
    # Graficar vectores de soporte
    if show_support and hasattr(clf, 'support_vectors_'):
        ax.scatter(
            clf.support_vectors_[:, 0],
            clf.support_vectors_[:, 1],
            s=150,
            linewidth=2,
            facecolors="none",
            edgecolors="red",
            label="Vectores de Soporte",
        )
    
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel('$x_0$', fontsize=12)
    ax.set_ylabel('$x_1$', fontsize=12)
    
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Mostrar precisión
    accuracy = accuracy_score(Y, clf.predict(X))
    ax.text(0.02, 0.98, f'Precisión: {accuracy:.3f}', 
            transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    return ax

## 1. Dataset Simple: Clasificación Arriba/Abajo

Comenzamos con un dataset simple donde la clase está determinada por el signo del eje Y.

In [ ]:
# ============================================================================
# 1.1 Generación y Visualización del Dataset
# ============================================================================
X, Y = load_dataset_up_down(100, seed=39)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X[:, 0], X[:, 1], c=Y, s=60, cmap=plt.cm.Spectral, 
                     edgecolors='k', linewidth=0.0, alpha=0.8)
ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel('$x_0$', fontsize=12)
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_title('Dataset Arriba/Abajo - Clases separables linealmente', fontsize=14, fontweight='bold')
#plt.colorbar(scatter, ax=ax, label='Clase')
plt.show()

print(f"Número de muestras: {len(X)}")
print(f"Distribución de clases: Clase 0 = {sum(Y==0)}, Clase 1 = {sum(Y==1)}")

### 1.2 Entrenamiento del SVM Lineal


In [ ]:
# Crear y entrenar el clasificador
clf = LinearSVC(C=1.0, loss='hinge', random_state=42)
clf.fit(X, Y)

# Extraer coeficientes para visualizar la frontera
w = clf.coef_[0]
a = -w[0] / w[1]
b = -clf.intercept_[0] / w[1]

# Generar puntos para la línea de decisión
x_0 = np.linspace(min(X[:, 0]) - 1, max(X[:, 0]) + 1, 100)
x_1 = a * x_0 + b

# Visualización
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x_0, x_1, 'r-', linewidth=2, label='Frontera de Decisión')
ax.scatter(X[:, 0], X[:, 1], c=Y, s=60, cmap=plt.cm.Spectral, 
           edgecolors='k', linewidth=0.0, alpha=0.8)
ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel('$x_0$', fontsize=12)
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_title('SVM Lineal - Frontera de Decisión', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')

# Mostrar ecuación de la frontera
eq_text = f'$x_1 = {a:.4f}x_0 + {b:.4f}$'
#ax.text(0.02, 0.98, eq_text, transform=ax.transAxes, fontsize=12,
ax.text(0.02, 0.1, eq_text, transform=ax.transAxes, fontsize=12,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.show()

# Métricas
predictions = clf.predict(X)
accuracy = accuracy_score(Y, predictions)
print(f"Precisión en entrenamiento: {accuracy:.3f}")
print(f"Coeficientes: w = {w}")
print(f"Intercept: {clf.intercept_[0]:.4f}")

## 2. Escalado y Dataset Más Grande

Evaluamos el rendimiento con más datos para ver la capacidad de generalización.

### 2.1 Dataset más grande

In [ ]:

X_large, Y_large = load_dataset_up_down(800, seed=39)

# Usar el mismo clasificador entrenado anteriormente
fig, ax = plt.subplots(figsize=(8, 6))
plot_svm_decision_boundary(clf, X_large, Y_large, title='SVM Lineal en Dataset Grande', ax=ax)
plt.tight_layout()
plt.show()

# Calcular precisión
predictions = clf.predict(X_large)
accuracy = accuracy_score(Y_large, predictions)
print(f"Precisión en dataset grande: {accuracy:.3f}")

## 3. Robustez al Ruido

Un aspecto importante de los modelos es su capacidad para manejar datos ruidosos. Veamos cómo afecta el ruido al SVM lineal.

### 3.1 Dataset con Ruido

In [ ]:
# Generar datos con diferentes niveles de ruido
noise_levels = [0.03, 0.10, 0.20]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, noise in enumerate(noise_levels):
    X_noise, Y_noise = load_dataset_up_down(800, seed=1, noise_ratio=noise)
    ax = axes[idx]
    ax.scatter(X_noise[:, 0], X_noise[:, 1], c=Y_noise, s=30, 
               cmap=plt.cm.Spectral, alpha=0.7)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_title(f'Ruido: {noise*100:.0f}%', fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_0$')
    ax.set_ylabel('$x_1$')

plt.tight_layout()
plt.show()

### 3.2 Entrenamiento con Ruido

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, noise in enumerate(noise_levels):
    X_noise, Y_noise = load_dataset_up_down(800, seed=1, noise_ratio=noise)
    
    # Entrenar SVM
    clf_noise = LinearSVC(C=1.0, loss='hinge', random_state=42)
    clf_noise.fit(X_noise, Y_noise)
    
    # Visualizar
    ax = axes[idx]
    plot_svm_decision_boundary(clf_noise, X_noise, Y_noise, 
                               title=f'Ruido: {noise*100:.0f}%', ax=ax)

plt.tight_layout()
plt.show()

### Observaciones sobre el Ruido

- **Ruido bajo (3%)**: El SVM mantiene una buena precisión y la frontera de decisión es estable.
- **Ruido medio (10%)**: Comienza a verse afectado, pero aún tiene buen rendimiento.
- **Ruido alto (20%)**: El modelo se degrada significativamente y la frontera de decisión se vuelve menos confiable.

## 4. Efecto del Parámetro C en Datos Superpuestos

El parámetro C controla la regularización. Veamos su efecto en un dataset donde las clases no son perfectamente separables.
### Parámetro C

El parámetro C controla el trade-off entre:
- **Margen amplio**: Generalización (C pequeño)
- **Clasificación correcta**: Ajuste a los datos (C grande)

La función de pérdida que se minimiza es:

$\lambda({C}) ||w||^2 + \frac{1}{n}\sum_{i=1}^n \max(0, 1-y_i (\mathbf{w^T x_i}-b))$

### 4.1 Dataset con Superposición

In [ ]:
X_blob, Y_blob = make_blobs(n_samples=200, centers=2, random_state=3, cluster_std=2.5)

# Dividir en entrenamiento y prueba
X_train, X_test, Y_train, Y_test = train_test_split(
    X_blob, Y_blob, test_size=0.3, random_state=42
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_train[:, 0], X_train[:, 1], c=Y_train, s=50, 
            cmap=plt.cm.Spectral, edgecolors='k', linewidth=0.0)
ax1.set_title('Datos de Entrenamiento', fontweight='bold')
ax1.set_xlabel('$x_0$')
ax1.set_ylabel('$x_1$')

ax2.scatter(X_test[:, 0], X_test[:, 1], c=Y_test, s=50, 
            cmap=plt.cm.Spectral, edgecolors='k', linewidth=0.5)
ax2.set_title('Datos de Prueba', fontweight='bold')
ax2.set_xlabel('$x_0$')
ax2.set_ylabel('$x_1$')

plt.tight_layout()
plt.show()

### 4.2 Comparación de Diferentes Valores de C

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_scores = []
test_scores = []
n_support_vectors = []

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

for idx, C in enumerate(C_values):
    # Entrenar SVM con el parámetro C
    clf_c = LinearSVC(C=C, loss='hinge', random_state=42, max_iter=10000)
    clf_c.fit(X_train, Y_train)
    
    # Guardar métricas
    train_score = accuracy_score(Y_train, clf_c.predict(X_train))
    test_score = accuracy_score(Y_test, clf_c.predict(X_test))
    
    train_scores.append(train_score)
    test_scores.append(test_score)
    
    # Visualizar en subplot
    ax = axes[idx]
    plot_svm_decision_boundary(clf_c, X_train, Y_train, 
                               title=f'C = {C}', ax=ax, show_support=False)
    
    # Añadir métricas
    ax.text(0.02, 0.90, f'Train: {train_score:.3f}', transform=ax.transAxes, 
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax.text(0.02, 0.82, f'Test: {test_score:.3f}', transform=ax.transAxes, 
            fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    decision_function = clf_c.decision_function(X_train)
    support_vector_indices = np.where(np.abs(decision_function) <= 1 + 1e-15)[0]
    support_vectors = X_train[support_vector_indices]
    plt.scatter(
        support_vectors[:, 0],
        support_vectors[:, 1],
        s=100,
        linewidth=1,
        facecolors="none",
        edgecolors="k",
    )

# Ocultar subplot extra
for idx in range(len(C_values), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
X = X_train
Y = Y_train
plt.figure(figsize=(10, 5))
for i, C in enumerate([0.01, 200]):
    # "hinge" is the standard SVM loss
    clf = LinearSVC(C=C, loss="hinge", random_state=26).fit(X, Y)
    predictions = clf.predict(X)
    print (C, '- Accuracy: %d ' % ((np.sum(Y == predictions))/float(Y.size)*100))
    # obtain the support vectors through the decision function
    decision_function = clf.decision_function(X)
    # we can also calculate the decision function manually
    # decision_function = np.dot(X, clf.coef_[0]) + clf.intercept_[0]
    # The support vectors are the samples that lie within the margin
    # boundaries, whose size is conventionally constrained to 1
    support_vector_indices = np.where(np.abs(decision_function) <= 1 + 1e-15)[0]
    support_vectors = X[support_vector_indices]

    plt.subplot(1, 2, i + 1)
    plt.scatter(X[:, 0], X[:, 1], c=Y, s=30, cmap=plt.cm.Spectral)
    ax = plt.gca()
    DecisionBoundaryDisplay.from_estimator(
        clf,
        X,
        ax=ax,
        grid_resolution=50,
        plot_method="contour",
        colors="k",
        levels=[-1, 0, 1],
        alpha=0.5,
        linestyles=["--", "-", "--"],
    )
    plt.scatter(
        support_vectors[:, 0],
        support_vectors[:, 1],
        s=100,
        linewidth=1,
        facecolors="none",
        edgecolors="k",
    )
    plt.title("C=" + str(C))
plt.tight_layout()
plt.show()

### 4.3 Análisis del Efecto de C

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de precisión vs C
ax1.plot(C_values, train_scores, 'o-', label='Entrenamiento', linewidth=2, markersize=8)
ax1.plot(C_values, test_scores, 's-', label='Prueba', linewidth=2, markersize=8)
ax1.set_xscale('log')
ax1.set_xlabel('C (escala logarítmica)', fontsize=12)
ax1.set_ylabel('Precisión', fontsize=12)
ax1.set_title('Precisión vs Parámetro C', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico de margen vs C (visualizado a través de la diferencia entre train y test)
overfitting_gap = np.array(train_scores) - np.array(test_scores)
ax2.bar(range(len(C_values)), overfitting_gap, tick_label=C_values)
ax2.set_xlabel('C', fontsize=12)
ax2.set_ylabel('Sobreajuste (Train - Test)', fontsize=12)
ax2.set_title('Sobreajuste vs Parámetro C', fontweight='bold')
ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resumen de resultados
print("="*60)
print("RESUMEN DEL EFECTO DEL PARÁMETRO C")
print("="*60)
for C, train, test in zip(C_values, train_scores, test_scores):
    print(f"C = {C:>6}: Train = {train:.3f}, Test = {test:.3f}, Gap = {train-test:.3f}")
print("="*60)
print("""
Observaciones:
- C pequeño → Margen amplio → Menor sobreajuste, pero puede subajustar
- C grande → Margen pequeño → Mayor sobreajuste, pero mejor ajuste en entrenamiento
- El valor óptimo de C depende del dataset y se encuentra mediante validación cruzada
""")

## 5. SVM para Datos No Balanceados

Cuando las clases están desbalanceadas, podemos usar pesos para compensar.

In [ ]:
LinearSVC?

### 5.1 Dataset Desbalanceado


In [ ]:
np.random.seed(42)
X_imb = np.random.randn(500, 2)
Y_imb = np.zeros(500, dtype=np.int8)
Y_imb[:100] = 1  # Clase 1 es minoritaria
X_imb[:100] += np.array([2, 2])

print(f"Distribución de clases: Clase 0 = {sum(Y_imb==0)}, Clase 1 = {sum(Y_imb==1)}")

plt.scatter(X_imb[:, 0], X_imb[:, 1], c=Y_imb, s=30, cmap=plt.cm.Spectral)
#plt.axis('equal')
plt.axis('square')

plt.show()


### 5.2 Entrenamiento


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clasificador sin balancear
clf_unbalanced = LinearSVC(C=1.0, random_state=42)
clf_unbalanced.fit(X_imb, Y_imb)

ax = axes[0]
plot_svm_decision_boundary(clf_unbalanced, X_imb, Y_imb, 
                          title='SVM sin Balanceo', ax=ax, show_support=False)
ax.text(0.02, 0.02, f'Clase 0: {sum(Y_imb==0)}, Clase 1: {sum(Y_imb==1)}', 
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Clasificador con balanceo
clf_balanced = LinearSVC(C=1.0, class_weight='balanced', random_state=42)
clf_balanced.fit(X_imb, Y_imb)

ax = axes[1]
plot_svm_decision_boundary(clf_balanced, X_imb, Y_imb, 
                          title='SVM con Balanceo (class_weight="balanced")', 
                          ax=ax, show_support=False)
ax.text(0.02, 0.02, f'Clase 0: {sum(Y_imb==0)}, Clase 1: {sum(Y_imb==1)}', 
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

# Comparación de precisión por clase
print("\n" + "="*60)
print("COMPARACIÓN DE PRECISIÓN POR CLASE")
print("="*60)
for name, clf in [('Sin balanceo', clf_unbalanced), ('Con balanceo', clf_balanced)]:
    pred = clf.predict(X_imb)
    print(f"\n{name}:")
    for clase in [0, 1]:
        mask = Y_imb == clase
        acc = accuracy_score(Y_imb[mask], pred[mask])
        print(f"  Clase {clase}: {acc:.3f} ({sum(mask)} muestras)")

## Referencias

1. [Scikit-learn: Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)
2. [Scikit-learn: SVC Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
3. [Scikit-learn: LinearSVC Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)